### **1.Setup**

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency

train_path=os.path.join("..","data","raw","train.csv")
clientes_path=os.path.join("..","data","raw","clientes.csv")

df_train= pd.read_csv(train_path,index_col="ID",sep=",")
df_clientes = pd.read_csv(clientes_path, sep=",")

### **2.Loading Data and First Look**

#### **2.1. Overall First Look**

In [ ]:
print(f"Rows:{df_train.shape[0]:,} | Columns:{df_train.shape[1]}")
df_train.head().T

In [1]:
#df_train.info() shows the type of each column
df_train.info()

NameError: name 'df_train' is not defined

Purchase Date as String is a problem

#### **2.2 Grouping the Variables**

To better organize our data we chose to group the variables by themes:

In [ ]:
#Variables with special structural roles
target_col = "repeat_purchase_90d"
group_col = "customer_id"
index_col = "ID"

# Grouping by themes
var_groups = {
    "Identifier": [
        "ID",
        "customer_id"
    ],
    "Target": [
        "repeat_purchase_90d"
    ],
    "Current transaction (financial)": [
        "Total a Pagar",
        "Total Bruto",
        "Total Liquido",
        "Total Imp_",
        "Total Portes",
        "Total Desc_ Global",
        "Total Desc_ Linha",
        "Custo"
    ],
    "Current transaction (operational)": [
        "N_ Detalhes",
        "invoice_count_current_purchase"
    ],
    "Transaction context & date": [
        "purchase_date",
        "payment_type",
        "salesperson",
        "commercial_zone",
        "warehouse",
        "document_series",
        "transport_method"
    ],
    "Customer historical summary": [
        "customer_tenure_days",
        "prior_purchase_count",
        "prior_total_spend",
        "prior_mean_purchase_value",
        "prior_std_purchase_value",
        "prior_min_purchase_value",
        "prior_max_purchase_value"
    ],
    "Customer cadence & gaps": [
        "days_since_previous_purchase",
        "prior_mean_gap_days",
        "prior_std_gap_days"
    ],
    "Rolling windows (RFM)": [
        "purchase_count_30d",
        "spend_30d",
        "purchase_count_90d",
        "spend_90d",
        "purchase_count_180d",
        "spend_180d",
        "purchase_count_365d",
        "spend_365d"
    ],
    "Synthetic / Control": [
        "random_noise"
    ]
}
pd.DataFrame(
    [(group, len(cols), ", ".join(cols)) for group, cols in var_groups.items()], columns=["Group", "# vars", "Variables"])

#### **2.3.The Target and the Floor**

In [ ]:
balance = df_train[target_col].value_counts().sort_index() #counting the number of 0 and 1 of the target column
print(balance.to_frame("rows").assign(share=lambda f: (f["rows"] / f["rows"].sum()).round(3)).to_string())

# The  floor: always predict 1 (F1 of class 1).
p = df_train[target_col].mean()
print(f"\nalways predict 1 -> accuracy {p:.3f} | F1(class 1) {2 * p / (1 + p):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
shares = df_train[target_col].value_counts(normalize=True).sort_index() * 100
ax.barh(["0 · no repeat", "1 · repeat"], shares.values, color=["#C96A00", "#1779A8"])
for i, v in enumerate(shares.values):
    ax.text(v + 1, i, f"{v:.1f}%", va="center")
ax.set(xlim=(0, 100), xlabel="% of purchase occasions", title="Target balance in train.csv")
plt.tight_layout()
plt.show()

**What this tells us**

* The classes are **almost balanced (class 1≈56%/class 0≈44%)** .
* Two consequences for later: (1) every F1 we report will be compared with the all-ones floor, and
  (2) the decision **threshold** is part of the model. The default 0.5 is not necessarily the one that maximises F1.

###  **3. Data Quality Assessment**

  Before changing anything, we first diagnose the problems.

#### **3.1. Rows with no information**

Before looking at each column, we look at each row, to see how much information each row gives us.

In [ ]:
row_missing_percentage = pd.DataFrame({
    'customer_id': df_train['customer_id'],
    'n_missing': df_train.isna().sum(axis=1),
    'pct_missing': (df_train.isna().sum(axis=1) * 100) / df_train.shape[1]})

row_missing_percentage = (
    row_missing_percentage
    .set_index("customer_id")
    .query("n_missing > 19")
    .sort_values("n_missing", ascending=False))

print(f"There are {row_missing_percentage.shape[0]} rows with more than 50% of missing values")
row_missing_percentage

In [ ]:
# rows where everything is missing except customer_id and the target
empty_rows = df_train.drop(columns=["customer_id", "repeat_purchase_90d"]).isna().all(axis=1)
print(f"Empty rows: {empty_rows.sum()}")

# df is our working copy without the empty rows (df_train stays untouched)
df = df_train.loc[~empty_rows]
print(f"Rows in df: {df.shape[0]:,}")

There are 65 rows completely empty (the ones we identified before), having only `customer_id` and the target. We **consider them disposable because they do not give us any information**, so from now on we analyse `df`, which is `df_train` without these 65 rows.

#### **3.2. Duplicates**

In [ ]:
n_dup_rows=df_train.duplicated().sum()  #rows that are 100% identical in the raw file
print(f"Fully duplicated rows (raw file): {n_dup_rows}")
print(f"Fully duplicated rows (without the empty rows): {df.duplicated().sum()}")

n_dup_ids=df["customer_id"].duplicated().sum() #repeated client IDs
n_client_ids= df["customer_id"].nunique()
print(f"Duplicated client_id  : {n_dup_ids}")
print(f"Number of clients : {n_client_ids}")
print(f"Number of receipts/clients: {(len(df)/n_client_ids):.2f}")

In the raw file there are 10 fully duplicated rows, but they are all empty rows: two empty rows of the same client look the same. Without the empty rows there are **no duplicated purchases**.

The repeated `customer_id` is not a problem: there are 6,469 purchases from 607 clients, about 10.7 purchases per client. This means the rows of the same client are not independent, which we have to keep in mind when we split the data for validation.

#### **3.3. Missing Values**

In [ ]:
missing_values=(df.isna().sum().to_frame("n_missing").assign(percentage=lambda x: (x["n_missing"]*100/len(df))).query("n_missing>0").sort_values("percentage", ascending=False))
missing_values

We have 12 columns with at least 190 missing values. Some of the missing values may be explained by the lack of history.Lets look at the column prior_pucharse_count and see how many missing values per clients with: no history (prior_pucharse_count=0), 1 prior purchase, 2 prior purchases and 3+ prior purchases.

In [ ]:
# Hypothesis: history columns are missing because the customer has (almost) no history.
history_missing = ["days_since_previous_purchase", "prior_mean_purchase_value", "prior_std_purchase_value",
                   "prior_min_purchase_value", "prior_mean_gap_days", "prior_std_gap_days"]
prior_n = df["prior_purchase_count"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})
table = (df[history_missing].isna()
         .groupby(prior_n).mean().mul(100).round(1)
         .assign(rows=prior_n.value_counts()))
table.index.name = "prior purchases"
table

The hypothesis holds **only in part**:

* With **0 prior purchases**, every history column is missing in 100% of rows, as expected: there is no history.
* With **1 prior purchase**, the gap and the standard deviations are always missing (one purchase has no gap and no
  spread), but `days_since_previous_purchase` should exist, and it is still missing for a fraction of rows.
* With **2 or more** prior purchases, all these columns *should* exist, yet a few percent are missing anyway. That
  part is ordinary missingness, not structural.

So "missing = no history" (the business-logic reading, MNAR) explains most but not all of it. This decides the fix: we
add an explicit flag such as `is_first_purchase` (`prior_purchase_count == 0`) and *missing indicators*, then impute
the remaining gaps with the **training median**. We do **not** fill gaps with 0: a gap of 0 days would mean "buys every
day", which is the opposite of "no history".

Without the empty rows, the invoice columns, `commercial_zone`, `document_series`, `purchase_date` and the rolling windows have **no missing values**, except `spend_365d` (195 missing values). The real missing values are only in the history columns, in `spend_365d` and in some context columns (about 3% each).

Some history columns have around 20% of missing values, but we do **not** drop them: the missing values are mostly structural (clients without previous purchases), and these columns are among the most important ones for the target (for example `prior_mean_gap_days`). Dropping them would lose a lot of information.

#### **3.4. Inconsistent Categories**

Lets verify if there are any categorical values with inconsistent values. Incosistent values would make the model consider values with the same meaning different due to typography errors.

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist() #columns with categorical values
cat_cols.remove("purchase_date")  # it is a date, not a category
for col in cat_cols:
    print(f"{col:<10} -> {df[col].nunique()} distinct values")

Each category is always written in the same way, so there are no spelling variants to fix. The real problem is the **rare levels**: some levels appear only a few times (for example `commercial_zone` 6 and 7 with 1 row each, and many `transport_method` levels with only 1 row). Lets look at each column.

In [ ]:
df["payment_type"].value_counts(dropna=False)

Payment Type has 195 Missing values and no inconsistencies. Values with <10 counts may be considered rare values.

In [ ]:
df[("salesperson")].value_counts(dropna=False)

Salesperson only has missing values.Values with <10 counts may be considered rare values.

In [ ]:
df[("commercial_zone")].value_counts(dropna=False)

This category is fine and has no missing values once the empty rows are removed. Zones 6 and 7 only have 1 row each, so they are rare levels.

In [ ]:
print(f"There are {df['transport_method'].isna().sum()} NaNs in this category")
df["transport_method"].value_counts(dropna=False)

This category has 196 missing values and values with one observation, so they will be considered rare values.

In [ ]:
df[("warehouse")].value_counts(dropna=False)

Excluding the missing values, this column is fine.

In [ ]:
df[("document_series")].value_counts(dropna=False)

#### **3.5. Wrong Data Types**

#### **3.5.1. Dates**

In [ ]:
# Date columns: check which formats are used by replacing every digit with "9"
patterns = df["purchase_date"].astype(str).str.replace(r"\d", "9", regex=True).value_counts()
print(f"{'date format':<18}", patterns.to_dict())
print(f"{'missing values':<18}", df["purchase_date"].isna().sum())

The dates are all in the format 9999-99-99, and there are no missing dates once the empty rows are removed.

#### **3.6. Impossible Values**

Lets analyse if there are impossible values "hidden" within the data.

In [ ]:
cols_num = df.select_dtypes(include=["number"]).columns.tolist()
df[cols_num].describe().T[["min","25%","50%","75%","max"]].round(2)

Two things cannot be true:

* `prior_mean_purchase_value` and `spend_365d` reach **1,000,000,000** (one billion euros), while their medians are
  ≈725 € and ≈2,400 €.
* `days_since_previous_purchase` and `prior_mean_gap_days` have **negative** values (a gap cannot be negative), and
  `prior_mean_purchase_value` and `spend_365d` too (spending cannot be negative).

Random damage would scatter over many values. Placeholders sit on a few exact values, so we look at the values themselves.

In [ ]:
checks = {
    "prior_mean_purchase_value": df["prior_mean_purchase_value"],
    "spend_365d": df["spend_365d"],
    "days_since_previous_purchase": df["days_since_previous_purchase"],
    "prior_mean_gap_days": df["prior_mean_gap_days"],
}
for name, s in checks.items():
    bad = s[(s < 0) | (s >= 1e8)]
    print(f"{name:30s} {len(bad):3d} impossible values -> {bad.value_counts().head(4).to_dict()}")

We have observed that the values 1e9 and -4 are exact repeated values, so they look like codes and not real values. They are impossible values considering the financial context. Before deciding what to do, we check if the real value can be recovered from other columns.

##### **3.6.1. Impossible Values of `prior_mean_purchase_value`**

Lets examine the negative values of `prior_mean_purchase_value`:

In [ ]:
df[df["prior_mean_purchase_value"]<0].T

We have noticed that the prior mean purchase value may be recomputed through the total spend and the purchase count. Lets confirm this theory:

In [ ]:
recomputed_mean = df["prior_total_spend"] / df["prior_purchase_count"].replace(0, np.nan)
stored_mean = df["prior_mean_purchase_value"]

is_billion = stored_mean >= 1e8
is_negative = stored_mean < 0
is_normal = stored_mean.notna() & ~is_billion & ~is_negative
pd.DataFrame({
    "prior_total_spend": df.loc[is_negative, "prior_total_spend"],
    "prior_purchase_count": df.loc[is_negative, "prior_purchase_count"],
    "total / count": recomputed_mean[is_negative],
    "stored prior_mean_purchase_value": stored_mean[is_negative],
}).head(5).round(2)

**The 16 negative values are a sign error**: in every one of them, the stored value is exactly
  `−(prior_total_spend / prior_purchase_count)`. There is no business logic behind a negative mean.

In [ ]:
pd.DataFrame({
    "rows": [is_normal.sum(), is_negative.sum(), is_billion.sum()],
    "stored = total / count": [np.isclose(stored_mean[m], recomputed_mean[m]).mean()
                               for m in (is_normal, is_negative, is_billion)],
    "|stored| = total / count": [np.isclose(stored_mean[m].abs(), recomputed_mean[m]).mean()
                                 for m in (is_normal, is_negative, is_billion)],
    "total / count can be computed": [recomputed_mean[m].notna().mean()
                                      for m in (is_normal, is_negative, is_billion)],
}, index=["normal values", "negative values", "1e9 values"]).round(3)

**The formula holds**: for 99.2% of the normal rows, the stored mean is exactly total / count.

**The 16 values of 1e9 can be recovered as well**: all these customers have previous purchases, so total / count can
  be computed for every one of them.

##### **3.6.2. And the negative values of `spend_365d`?**

There is no general formula to recompute `spend_365d` from other columns, but it must respect two bounds: it includes the last 180 days, so
it is at least `spend_180d`, and it is part of the whole history, so it is at most `prior_total_spend`. We check
whether the negative values fit these bounds once the sign is removed.

In [ ]:
spend = df["spend_365d"]
spend_negative = spend < 0
spend_billion = spend >= 1e8
spend_normal = spend.notna() & ~spend_negative & ~spend_billion


def inside_bounds(values, rows):
    """True where a 365-day spend lies between the 180-day spend and the full prior spend (1 cent tolerance)."""
    return ((values >= df.loc[rows, "spend_180d"] - 0.01)
            & (values <= df.loc[rows, "prior_total_spend"] + 0.01))


pd.DataFrame({
    "rows": [spend_normal.sum(), spend_negative.sum()],
    "value inside the bounds": [inside_bounds(spend[spend_normal], spend_normal).mean(),
                                inside_bounds(spend[spend_negative], spend_negative).mean()],
    "|value| inside the bounds": [inside_bounds(spend[spend_normal].abs(), spend_normal).mean(),
                                  inside_bounds(spend[spend_negative].abs(), spend_negative).mean()],
}, index=["normal spend_365d", "negative spend_365d"]).round(3)

In [ ]:
# The negative value that does not fit the bounds even without its sign
outside = spend_negative & ~inside_bounds(spend.abs(), df.index)
df.loc[outside, ["spend_180d", "spend_365d", "prior_total_spend"]]

* **14 of the 15 negative values fit the bounds once the sign is removed**, so a sign error is possible (99.4% of the normal values respect the same bounds).
* **One value cannot be explained**: −159,115 € in absolute value is three times the customer's whole prior spend
  (49,329 €). No correction is defensible, so it is treated as unknown.
* The 15 values of **1e9** in `spend_365d` cannot be reconstructed from other columns either (only in a few special cases, see 3.6.7).

But is fitting the bounds enough to trust the sign correction? We check how often a random `spend_365d` value would also fit the bounds of these rows.

In [ ]:
# for each of the 14 rows, how many normal spend_365d values (taken at random) would also fit its bounds?
pool = spend[spend_normal]
fits = spend_negative & inside_bounds(spend.abs(), df.index)
chance = []
for row in df.index[fits]:
    low, high = df.loc[row, "spend_180d"], df.loc[row, "prior_total_spend"]
    chance.append(((pool >= low - 0.01) & (pool <= high + 0.01)).mean())
print(f"On average {np.mean(chance):.0%} of random values would also fit the bounds")

About 45% of random values would also fit, because the bounds are wide. So the bounds do not prove that only the sign is wrong: the whole value could be a recording error. This is different from `prior_mean_purchase_value`, where the value is exactly equal to total / count, which a random value would almost never be.

So we do **not** just remove the minus sign: the 15 negative values of `spend_365d` will be treated as missing (NaN). The ones that can not be recovered (3.6.7) are imputed with the median in Part 2. The median has to be computed only on the training rows, inside the pipeline, to avoid leakage, and we use the median and not the mean because the column is very skewed. This affects the 15 negative values (69 values of `spend_365d` in total become missing, see 3.6.3).

##### **3.6.3. Values that look normal but are not**

The bounds above can also be checked on the "normal" values. A mean purchase value cannot be bigger than the biggest purchase of the client, and the spend of the last 365 days cannot be bigger than the whole prior spend.

In [ ]:
# mean bigger than the max purchase of the client
mean_too_high = is_normal & (stored_mean > df["prior_max_purchase_value"] + 0.01)
print(f"prior_mean_purchase_value bigger than prior_max_purchase_value: {mean_too_high.sum()}")
print(f"normal values that do not match total / count: {(is_normal & ~np.isclose(stored_mean, recomputed_mean)).sum()}")

# spend of the last 365 days bigger than the whole prior spend
spend_too_high = spend_normal & (spend > df["prior_total_spend"] + 0.01)
print(f"spend_365d bigger than prior_total_spend: {spend_too_high.sum()}")

# missing means that can be computed with total / count
print(f"prior_mean_purchase_value missing but computable: {(stored_mean.isna() & recomputed_mean.notna()).sum()}")

* 46 "normal" means do not match total / count, and in 45 of them the mean is bigger than the biggest purchase of the client, which is impossible. These values look fine alone, so `describe()` could not show them. Since total / count always works, we can recompute `prior_mean_purchase_value` for every row.
* 39 values of `spend_365d` are bigger than the whole prior spend of the client, also impossible. There is no general formula to recover them, so they will be treated as missing (some can still be computed in special cases, see 3.6.7).
* 194 missing values of `prior_mean_purchase_value` can be computed with total / count, so they are not really missing.

##### **3.6.4. Are both negatives on the same rows?**

We test the hypothesis that `prior_mean_purchase_value` and `spend_365d` are negative in the same rows. If this is true, the minus sign is probably a typing error that affects the whole row, and removing it would be easier to defend. If not, we keep the decision we took before.

In [ ]:
# rows where each column is negative, and rows where both are negative
neg_mean = df["prior_mean_purchase_value"] < 0
neg_spend = df["spend_365d"] < 0
print(f"Rows with negative prior_mean_purchase_value: {neg_mean.sum()}")
print(f"Rows with negative spend_365d: {neg_spend.sum()}")
print(f"Rows with both negative: {(neg_mean & neg_spend).sum()}")

df.loc[neg_mean | neg_spend, ["customer_id", "prior_mean_purchase_value", "spend_365d", "prior_total_spend", "prior_purchase_count"]]

The two columns are never negative in the same row. So the negative values are not an error of the whole row, and we keep the decision we took before: `prior_mean_purchase_value` is recomputed as total / count, and the negative values of `spend_365d` are treated as missing.

##### **3.6.5. Are the invoices consistent?**

The invoice columns are linked by accounting rules: the net value is the gross value minus the discounts, and the amount to pay is the net value plus the tax. If a row breaks these rules, one of its values is wrong. Lets check them.

In [ ]:
# the accounting rules of the invoice (2 cents of tolerance for the rounding)
net_rule = np.isclose(df["Total Liquido"], df["Total Bruto"] - df["Total Desc_ Linha"] - df["Total Desc_ Global"], atol=0.02)
pay_rule = np.isclose(df["Total a Pagar"], df["Total Liquido"] + df["Total Imp_"], atol=0.02)
print(f"Total Liquido = Total Bruto - discounts:    {net_rule.mean():.1%} of the rows")
print(f"Total a Pagar = Total Liquido + Total Imp_: {pay_rule.mean():.1%} of the rows")

The two rules hold in 100% of the rows, so the invoice columns have no errors to correct. It also means that these columns are built from each other, which we will see again in the redundancy analysis (section 8).

##### **3.6.6. Is the biggest purchase possible?**

In 3.6.3 we compared the mean with the biggest purchase of the client, but the biggest purchase also has a limit. A client with `count` prior purchases made `count − 1` purchases besides the biggest one, and each of them is at least the minimum. So the biggest one is at most `prior_total_spend − (count − 1) × prior_min_purchase_value`.

In [ ]:
count = df["prior_purchase_count"]
max_value = df["prior_max_purchase_value"]
max_limit = df["prior_total_spend"] - (count - 1) * df["prior_min_purchase_value"]
max_too_high = (count >= 1) & (max_value > max_limit + 0.01)

print(f"prior_max_purchase_value above its limit: {max_too_high.sum()}")
print(f"  of which bigger than the whole prior spend: {(max_too_high & (max_value > df['prior_total_spend'] + 0.01)).sum()}")
df.loc[max_too_high, ["prior_purchase_count", "prior_total_spend", "prior_min_purchase_value", "prior_max_purchase_value"]].head()

46 values are above this limit, and 45 of them are even bigger than the whole prior spend of the client: a client with 5 purchases and 3,456 € of total spend can not have one purchase of 22,844 €. Like the inflated means of 3.6.3, these values look normal alone, so `describe()` could not show them. They are impossible, so they will be treated as missing.

##### **3.6.7. Values that can be recovered**

In 3.6.1 the mean could be computed again as total / count. Some other columns are also built from each other, so a missing (or wrong) value is not always unknown:
* With 1 prior purchase, the previous purchase is also the first one, so `days_since_previous_purchase` is the tenure of the client.
* The tenure is the days since the previous purchase plus all the gaps between the purchases, so `days_since = tenure − gap × (count − 1)`, and the other way round for the gap.
* `spend_365d` is 0 if there was no purchase in the last 365 days, it is equal to `spend_180d` if there was no purchase between 180 and 365 days ago, and it is equal to `prior_total_spend` if all the prior purchases are in the last 365 days.
* With 1 prior purchase the biggest purchase is the total, and with 2 it is the total minus the minimum.

A rule is only useful if it always holds. So we first set the impossible values of 3.6 to missing, then we check each rule on the rows where we know both sides, and we count how many missing cells it can fill.

In [ ]:
# a copy with the impossible values of 3.6 set to missing (the mean is already covered in 3.6.1)
clean = df.copy()
bad_spend = (clean["spend_365d"] < 0) | (clean["spend_365d"] >= 1e8) | (clean["spend_365d"] > clean["prior_total_spend"] + 0.01)
clean.loc[bad_spend, "spend_365d"] = np.nan
for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:
    clean.loc[clean[col] < 0, col] = np.nan
clean.loc[max_too_high, "prior_max_purchase_value"] = np.nan

count, tenure = clean["prior_purchase_count"], clean["customer_tenure_days"]
days, gap, spend = clean["days_since_previous_purchase"], clean["prior_mean_gap_days"], clean["spend_365d"]

# rule: (rows where it applies, stored value, value given by the rule)
rules = {
    "days_since = tenure (1 prior purchase)": (count == 1, days, tenure),
    "days_since = tenure - gap x (count - 1)": (count >= 2, days, tenure - gap * (count - 1)),
    "gap = (tenure - days_since) / (count - 1)": (count >= 2, gap, (tenure - days) / (count - 1)),
    "spend_365d = 0 (no purchase in 365 days)": (clean["purchase_count_365d"] == 0, spend, pd.Series(0.0, index=clean.index)),
    "spend_365d = spend_180d (no purchase 180-365 days)": (clean["purchase_count_365d"] == clean["purchase_count_180d"], spend, clean["spend_180d"]),
    "spend_365d = prior_total_spend (all purchases in 365 days)": (clean["purchase_count_365d"] == count, spend, clean["prior_total_spend"]),
    "max = total (1 prior purchase)": (count == 1, clean["prior_max_purchase_value"], clean["prior_total_spend"]),
    "max = total - min (2 prior purchases)": (count == 2, clean["prior_max_purchase_value"], clean["prior_total_spend"] - clean["prior_min_purchase_value"]),
}
table = []
for rule, (applies, stored, formula) in rules.items():
    known = applies & stored.notna() & formula.notna()
    table.append({"rule": rule, "rows to check": known.sum(),
                  "rule holds": f"{np.isclose(stored[known], formula[known], atol=0.01).mean():.1%}",
                  "missing cells it can fill": (applies & stored.isna() & formula.notna()).sum()})
pd.DataFrame(table).set_index("rule")

Every rule holds in 100% of the rows we can check, so these rules are not estimates: they give the true values. The function below applies them (and total / count of 3.6.1), using only the values of the same row.

In [ ]:
def recover_values(frame):
    """Fills the cells that can be computed exactly from other columns of the same row (3.6.1 and 3.6.7)."""
    out = frame.copy()
    count, tenure = out["prior_purchase_count"], out["customer_tenure_days"]
    out["prior_mean_purchase_value"] = out["prior_total_spend"] / count.replace(0, np.nan)

    days, gap = out["days_since_previous_purchase"].copy(), out["prior_mean_gap_days"].copy()
    fill = days.isna() & (count == 1)
    out.loc[fill, "days_since_previous_purchase"] = tenure[fill]
    fill = days.isna() & (count >= 2) & gap.notna()
    out.loc[fill, "days_since_previous_purchase"] = (tenure - gap * (count - 1))[fill]
    fill = gap.isna() & (count >= 2) & days.notna()
    out.loc[fill, "prior_mean_gap_days"] = ((tenure - days) / (count - 1))[fill]

    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == 0)
    out.loc[fill, "spend_365d"] = 0.0
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == out["purchase_count_180d"])
    out.loc[fill, "spend_365d"] = out.loc[fill, "spend_180d"]
    fill = out["spend_365d"].isna() & (out["purchase_count_365d"] == count)
    out.loc[fill, "spend_365d"] = out.loc[fill, "prior_total_spend"]

    fill = out["prior_max_purchase_value"].isna() & (count == 1)
    out.loc[fill, "prior_max_purchase_value"] = out.loc[fill, "prior_total_spend"]
    fill = out["prior_max_purchase_value"].isna() & (count == 2)
    out.loc[fill, "prior_max_purchase_value"] = (out["prior_total_spend"] - out["prior_min_purchase_value"])[fill]
    return out


recovered = recover_values(clean)
cols = ["days_since_previous_purchase", "prior_mean_gap_days", "spend_365d", "prior_max_purchase_value"]
print(f"Missing cells recovered: {(clean[cols].isna().sum() - recovered[cols].isna().sum()).sum()}")
wrong = {"days_since_previous_purchase": df["days_since_previous_purchase"] < 0,
         "prior_mean_gap_days": df["prior_mean_gap_days"] < 0,
         "spend_365d": bad_spend,
         "prior_max_purchase_value": max_too_high}
pd.DataFrame({
    "missing in the file": df[cols].isna().sum(),
    "after removing the impossible values": clean[cols].isna().sum(),
    "after recovering": recovered[cols].isna().sum(),
    "impossible values recovered": [(wrong[c] & recovered[c].notna()).sum() for c in cols],
    "impossible values": [wrong[c].sum() for c in cols],
})

Removing the impossible values first creates more missing values, but the rules recover many more cells than that: 516 in total. Most of the gaps of -4 days come back (15 of 16 for `days_since_previous_purchase` and 14 of 16 for `prior_mean_gap_days`), and also 23 of the 69 wrong values of `spend_365d` and 4 of the 46 impossible maximums. The missing values that are left are the ones that really can not be computed, mostly the first purchases, which have no history (3.3).

These rules use only the values of the same row, so they can be applied to the whole file, and later to test.csv, without leakage.

##### **3.6.8. A cost of zero**

The `describe()` at the start of 3.6 shows that `Custo` has a minimum of 0. A sale with no cost can be real (a service, an offer) or it can be a cost that was never recorded and was saved as 0. Lets look at these purchases.

In [ ]:
zero_cost = df["Custo"] == 0
print(f"Purchases with Custo = 0: {zero_cost.sum()} ({zero_cost.mean():.1%})")
print(f"Repeat rate | Custo = 0: {df.loc[zero_cost, target_col].mean():.2f} | Custo > 0: {df.loc[~zero_cost, target_col].mean():.2f}")
print(f"Total Liquido of these purchases (median): {df.loc[zero_cost, 'Total Liquido'].median():.0f} €")
print(f"Years with Custo = 0: {pd.to_datetime(df.loc[zero_cost, 'purchase_date']).dt.year.value_counts().sort_index().to_dict()}")

486 purchases (7.5%) have a cost of 0. They are not a period of the system (they appear in every year) and they are not tiny invoices (median of 187 €), and these purchases come back more than the others (0.68 vs 0.55). A cost of 0 is possible, so unlike the other values of 3.6 we can not call it impossible: we keep it as it is. In Part 2 we test if treating it as missing would help the model.

#### **3.7. Summary of the data quality problems**

| Problem | Rows | What we will do (Part 2) |
|---|---|---|
| Empty rows (only `customer_id` and target) | 65 | drop |
| Duplicated rows | 10 in the raw file, all empty rows; 0 after removing them | nothing |
| Missing values in the history columns (no previous purchase) | 600 to 1,400 per column | flag for first purchase + impute with the train median |
| Missing values in the context columns | about 195 per column | new category "Missing" |
| Rare categories | many in `transport_method` | group into "Other" |
| `purchase_date` as text | all | convert to datetime |
| `prior_mean_purchase_value` negative, 1e9, inflated or missing but computable | 16 + 16 + 46 + 194 | recompute as total / count |
| `spend_365d` negative (sign error not provable), 1e9 or bigger than the prior spend | 15 + 15 + 39 | missing, recover what the rules of 3.6.7 allow, then impute the rest with the train median |
| Gaps of -4 days | 16 + 16 | missing, recover what the rules of 3.6.7 allow, then impute the rest |
| `prior_max_purchase_value` above its possible limit | 46 | missing, recover what the rules of 3.6.7 allow, then impute the rest |
| Missing values that can be computed exactly from other columns | 516 (plus the 194 means of 3.6.3) | recover with the rules of 3.6.7, before any imputation |
| Invoice columns | all | consistent (3.6.5), nothing to do |
| `Custo` = 0 | 486 | keep (a possible value), test in Part 2 |

### **4.Outliers Values**

In money data a very big value can be an error or simply a very good client. The errors (1e9, wrong signs, gaps of -4 days) were already found in 3.6. For the other extreme values, lets check with the 1.5 x IQR rule how many rows would be called outliers, and if these clients buy again or not.

In [ ]:
# How many rows would the 1.5 x IQR rule call outliers, and do those clients buy again?
for col in ["Total a Pagar", "prior_total_spend", "prior_purchase_count", "N_ Detalhes"]:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    out = s > q3 + 1.5 * (q3 - q1)
    print(f"{col:22s} {100 * out.mean():.1f}% flagged | repeat rate flagged {df.loc[out[out].index, 'repeat_purchase_90d'].mean():.2f}"
          f" vs rest {df.loc[out[~out].index, 'repeat_purchase_90d'].mean():.2f}")

The rule flags between 7% and 10% of the rows, and these clients buy again much more than the rest (for example 0.97 vs 0.53 for `prior_purchase_count`). They are our best clients, not errors, so we keep them.

So we do **not** remove the outliers. The impossible values (1e9, wrong signs, -4 days) were already treated in 3.6 as errors, but the other extreme values are real. Since these variables are very skewed, in Part 2 we will use a `log1p` transformation for the models that are sensitive to the scale (the trees are not).

### **5. Time Structure**

Every purchase has a date, and the model will always be used to predict purchases that happen after the ones it learned from. Lets see how the data changes over time.

In [ ]:
# convert the date (it was read as string, see 3.5.1)
purchase_dates = pd.to_datetime(df["purchase_date"]) # converting the date from string to date format
print(f"First purchase: {purchase_dates.min().date()} | Last purchase: {purchase_dates.max().date()}")

# number of purchases and repeat rate per year
by_year = df.groupby(purchase_dates.dt.year)[target_col].agg(["count", "mean"])
by_year.columns = ["rows", "repeat rate"]
by_year.index.name = "year"
by_year.round(3)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(by_year.index, by_year["repeat rate"], marker="o")
plt.axhline(df[target_col].mean(), color="orange", linestyle="--", label=f"overall {df[target_col].mean():.2f}")
plt.ylim(0.3, 0.8)
plt.xlabel("year")
plt.ylabel("repeat rate")
plt.title("Repeat rate per year")
plt.legend()
plt.show()

The repeat rate is not the same every year: it goes from 0.48 (2022) to 0.63 (2017 and 2025), and the last years are above the overall average. This means the population changes over time (population drift), so the clients the model will see in the future are not exactly the same as the ones it learned from. 2025 only has 78 rows because the data ends in February 2025.

In [ ]:
# how many of the recent purchases come from clients we already knew?
recent = purchase_dates >= "2023-01-01"
old_clients = set(df.loc[~recent, "customer_id"])
recent_clients = set(df.loc[recent, "customer_id"])

print(f"Purchases from 2023 onwards: {recent.sum():,}")
print(f"Made by clients who already bought before 2023: {df.loc[recent, 'customer_id'].isin(old_clients).mean():.1%}")
print(f"Clients active since 2023: {len(recent_clients)} | new clients: {len(recent_clients - old_clients)}")

Almost 90% of the recent purchases come from clients that already bought before. Clients keep coming back over many years, so a future purchase will usually belong to a client that already has history in the data.

In [ ]:
# median of some variables in 3 periods
period = pd.cut(purchase_dates.dt.year, bins=[2014, 2018, 2022, 2025], labels=["2015-2018", "2019-2022", "2023-2025"])
cols_time = ["Total a Pagar", "N_ Detalhes", "days_since_previous_purchase", "purchase_count_90d", "spend_90d",
             "purchase_count_365d", "customer_tenure_days", "prior_purchase_count", "prior_total_spend"]
df.groupby(period, observed=True)[cols_time].median().T.round(1)

* The **cumulative variables grow with time** just because the company and the clients get older: the median tenure goes from 309 to 2,082 days and the prior total spend from 2,276 € to 15,014 €. The money values also go up (the median invoice goes from 382 € to 634 €).
* The **behaviour variables stay stable**: the days since the previous purchase (around 50-60) and the number of purchases in the last 90 or 365 days almost do not change.

**What we take from this:**
* The model will always predict the future, so the validation should also be **temporal**: each model is trained on older purchases and validated on the purchases that come after them, never before. A random split would train on future purchases to predict past ones, and the results would look better than they really are. We also leave a gap of 90 days between the train rows and the validation rows, because the target of a purchase looks 90 days ahead.
* We will **not use the year** of the purchase as a variable, because any future purchase will be in a year the model never saw.
* The cumulative variables will always be a bit higher in the future, so the behaviour variables (recency, frequency in fixed windows) are the safer ones.

### **6.Univariate Analysis**

For this part we use a copy of `df` with the corrections we decided in 3.6, so the wrong values do not change the statistics, and with the values that can be recovered exactly (3.6.7). The other missing values are **not** filled here (that is done in Part 2), so every statistic only uses the values that exist.

In [ ]:
# exploration copy with the corrections of 3.6 (df stays the same)
df_eda = df.copy()

# prior_mean_purchase_value is total / count, so we recompute it for every row
df_eda["prior_mean_purchase_value"] = df_eda["prior_total_spend"] / df_eda["prior_purchase_count"].replace(0, np.nan)

# spend_365d: 1e9, negative or bigger than the whole prior spend -> missing
bad_spend = (df_eda["spend_365d"] >= 1e8) | (df_eda["spend_365d"] < 0) | (df_eda["spend_365d"] > df_eda["prior_total_spend"] + 0.01)
df_eda.loc[bad_spend, "spend_365d"] = np.nan

# gaps of -4 days -> missing
for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:
    df_eda.loc[df_eda[col] < 0, col] = np.nan

# biggest purchase above its possible limit (3.6.6) -> missing
df_eda.loc[max_too_high, "prior_max_purchase_value"] = np.nan

# missing values that can be computed exactly (3.6.7)
missing_before = df_eda.isna().sum().sum()
df_eda = recover_values(df_eda)

print(f"spend_365d values set to missing: {bad_spend.sum()}")
print(f"Missing cells recovered: {missing_before - df_eda.isna().sum().sum()}")

#### **6.1. Numeric Variables**

In [ ]:
# numeric columns, taken from the groups of 2.2
numeric_cols = (var_groups["Current transaction (financial)"] + var_groups["Current transaction (operational)"]
                + var_groups["Customer historical summary"] + var_groups["Customer cadence & gaps"]
                + var_groups["Rolling windows (RFM)"])

shape = df_eda[numeric_cols].describe().T[["mean", "50%", "std", "max"]]
shape["mean/median"] = shape["mean"] / shape["50%"].replace(0, np.nan)
shape["skew"] = df_eda[numeric_cols].skew()
shape["% zeros"] = 100 * (df_eda[numeric_cols] == 0).mean()
shape.sort_values("skew", ascending=False).round(2)

Almost all the money and count variables are **very skewed to the right** (skew > 2 and the mean much bigger than the median): many small values and a long tail of very big ones. `Total Portes` and `Total Desc_ Global` are 0 in almost every row (84% and 98%), and `invoice_count_current_purchase` is almost always 1.

To be able to see the shape of these variables, the histograms of the very skewed ones (skew > 2) are shown with `log1p`. This is only for the plot, the statistics above use the original values. We use `log1p` and not `log` because many variables have zeros and log(0) does not exist.

In [ ]:
skewed = shape.index[shape["skew"] > 2].tolist()

fig, axes = plt.subplots(7, 4, figsize=(16, 20))
for ax, col in zip(axes.flat, numeric_cols):
    values = df_eda[col].dropna()
    if col in skewed:
        values = np.log1p(values.clip(lower=0))
        ax.set_title(f"log1p {col}", fontsize=10)
    else:
        ax.set_title(col, fontsize=10)
    ax.hist(values, bins=40, edgecolor="white", linewidth=0.5)
for ax in axes.flat[len(numeric_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

* With `log1p` most of the money variables look almost like a normal curve, so a `log1p` transformation can help the linear model in Part 2.
* The rolling windows have a lot of zeros (67% of the rows have no purchase in the last 30 days). Here a zero is information: it means no recent purchase.
* The extreme values of these variables are not errors: in section 4 we saw that they are our best clients, so we keep them.

#### **6.2. Categorical Variables**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, col in zip(axes.flat, cat_cols):
    counts = df_eda[col].fillna("<missing>").value_counts()
    top = counts.head(10)
    ax.barh(top.index[::-1], top.values[::-1])
    # write the number and the % next to each bar (small levels are not visible otherwise)
    for i, v in enumerate(top.values[::-1]):
        ax.text(v, i, f" {v:,} ({100 * v / counts.sum():.1f}%)", va="center", fontsize=8)
    title = f"{col} ({counts.size} levels" + (", top 10 shown)" if counts.size > 10 else ")")
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, counts.max() * 1.3)
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

* `warehouse` (Warehouse 1 in 99.7% of the rows with a value) and `commercial_zone` (97% in Zone 4) are almost constant, so they can not say much about the target.
* `Final Excel Warehouse 3` only appears in 21 purchases of 13 clients, all between July 2015 and March 2016. After that every purchase comes from Warehouse 1, so this level marks a period and not a type of client.
* `transport_method` has a long tail: a few methods have most of the rows and many have only a few rows.
* `payment_type` and `salesperson` have one main level but are more spread.

### **7. Relationship with the Target**

#### **7.1. Pearson or Spearman?**

Before measuring the relation of each numeric variable with the target, we have to choose which correlation to use, and then use the same one for all the variables:
* **Pearson** measures **linear** relations (the repeat rate goes up always by the same amount when the variable goes up) and is very sensitive to extreme values.
* **Spearman** uses the ranks of the values, so it measures any **monotonic** relation (always up or always down, even if not in a straight line) and is not affected by the extreme values.

The target is 0/1, so a normal scatter plot would only show two lines of points. Instead, we cut each variable in 10 groups with the same number of rows (deciles) and plot the repeat rate of each group against the median value of the group. If the points make a straight line the relation is linear, if they go up or down but curve it is monotonic.

We use equal-depth and not equal-width because our variables are very skewed: with equal-width almost all the rows would fall in the first interval and the other ones would be almost empty. With 10 groups each one has about 650 rows, so the repeat rate of each group is stable. This discretization is only used to look at the relations, the variables themselves are not changed here.

In [ ]:
# repeat rate in 10 groups of each variable (deciles)
fig, axes = plt.subplots(7, 4, figsize=(16, 22))
for ax, col in zip(axes.flat, numeric_cols):
    pair = df_eda[[col, target_col]].dropna()
    # we cut the rank and not the value, so variables with many equal values (like zeros) also get 10 groups
    groups = pd.qcut(pair[col].rank(method="first"), 10, labels=False)
    by_group = pair.groupby(groups).agg(median=(col, "median"), rate=(target_col, "mean"))
    ax.plot(by_group["median"], by_group["rate"], marker="o")
    ax.axhline(df_eda[target_col].mean(), color="orange", linestyle="--", linewidth=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_ylim(0, 1)
for ax in axes.flat[len(numeric_cols):]:
    ax.set_visible(False)
fig.supxlabel("median value of the group")
fig.supylabel("repeat rate")
plt.tight_layout()
plt.show()

* Almost no variable makes a straight line. The most important ones are **monotonic but curved**: the count and spend variables (`prior_purchase_count`, `prior_total_spend`, `purchase_count_365d`, `spend_365d`...) go up fast in the first groups and then become almost flat, and `prior_mean_gap_days` and `days_since_previous_purchase` go down until about 120-150 days and then stay flat.
* The values of the current invoice (`Total a Pagar`, `Custo`, `N_ Detalhes`) are almost flat and not even monotonic (they go a bit down and then up), so their correlation will be close to 0 with any method.
* So the relations are monotonic but not linear, and in 6.1 we saw that the variables have many extreme values. Both things are a problem for Pearson but not for Spearman. Lets confirm it with the numbers.

In [ ]:
corr_rows = []
for col in numeric_cols:
    pair = df_eda[[col, target_col]].dropna()
    corr_rows.append({"feature": col,
                      "Pearson": pair[col].corr(pair[target_col], method="pearson"),
                      "Spearman": pair[col].corr(pair[target_col], method="spearman")})

compare = pd.DataFrame(corr_rows).set_index("feature")
compare["|Spearman| - |Pearson|"] = compare["Spearman"].abs() - compare["Pearson"].abs()
print(f"Spearman is bigger than Pearson in {(compare['|Spearman| - |Pearson|'] > 0).sum()} of {len(compare)} variables")
compare.sort_values("|Spearman| - |Pearson|", ascending=False).round(3)

Spearman is bigger than Pearson in almost all the variables, and the difference is biggest exactly in the most useful ones: `prior_mean_gap_days` (−0.42 with Spearman vs −0.24 with Pearson), `days_since_previous_purchase` (−0.32 vs −0.20) or `purchase_count_365d` (0.44 vs 0.34). Pearson only gives a bigger value in the invoice values, where both are close to 0. Pearson underestimates the relations because they are curved and the extreme values pull the line.

**So we use the Spearman correlation for all the numeric variables**.

#### **7.2. Numeric Variables**

Following 7.1, we measure the relationship between each numeric variable and the target with the **Spearman** correlation, and we also show the median of each variable in each class.

In [ ]:
rel_rows = []
for col in numeric_cols:
    pair = df_eda[[col, target_col]].dropna()
    rho, p_value = stats.spearmanr(pair[col], pair[target_col])
    rel_rows.append({"feature": col, "Spearman rho": rho, "p-value": p_value,
                     "median | 0": pair.loc[pair[target_col] == 0, col].median(),
                     "median | 1": pair.loc[pair[target_col] == 1, col].median()})

relevance = pd.DataFrame(rel_rows).set_index("feature")
relevance = relevance.reindex(relevance["Spearman rho"].abs().sort_values(ascending=False).index)
relevance.round(3)

In [ ]:
values = relevance["Spearman rho"]
colors = ["tab:blue" if v > 0 else "tab:orange" for v in values[::-1]]

plt.figure(figsize=(8, 8))
plt.barh(values.index[::-1], values.values[::-1], color=colors)
plt.axvline(0, color="grey", linewidth=0.8)
plt.xlabel("Spearman correlation with repeat_purchase_90d")
plt.title("Relevance of each numeric variable\n(blue: more repeats when higher, orange: fewer)")
plt.tight_layout()
plt.show()

* **Recency and frequency are the strongest signals.** The clients who bought more times in the last 90, 180 and 365 days come back more (for example `purchase_count_365d`: median of 2 purchases for class 0 and 5 for class 1), and the clients with smaller gaps between purchases also come back more (`prior_mean_gap_days`: median of 103 days for class 0 and 56 for class 1). This is the RFM logic (Recency, Frequency, Monetary).
* **The values of the current invoice matter very little** (`Total a Pagar`, `Total Bruto`, `Total Liquido` have correlations close to 0).
* `customer_tenure_days` is weak: being an old client is not the same as being an active client.
* With 6,469 rows almost every p-value is very small, so we look at the size of the correlation and not at the p-value.

#### **7.3. Recency and Frequency**

In [ ]:
bins = [0, 15, 30, 60, 90, 180, 365, np.inf]
labels = ["0-15", "16-30", "31-60", "61-90", "91-180", "181-365", ">365"]
recency = pd.cut(df_eda["days_since_previous_purchase"], bins=bins, labels=labels, include_lowest=True)
recency = recency.cat.add_categories("none /\nmissing").fillna("none /\nmissing")
by_recency = df_eda.groupby(recency, observed=True)[target_col].agg(["mean", "size"])

plt.figure(figsize=(10, 4.5))
plt.bar(by_recency.index.astype(str), by_recency["mean"])
plt.axhline(df_eda[target_col].mean(), color="orange", linestyle="--", label=f"overall {df_eda[target_col].mean():.2f}")
for i, (rate, n) in enumerate(zip(by_recency["mean"], by_recency["size"])):
    plt.text(i, rate + 0.01, f"{rate:.2f}\nn={n}", ha="center", fontsize=8)
plt.ylim(0, 1)
plt.xlabel("days since the previous purchase")
plt.ylabel("repeat rate")
plt.title("Repeat rate by days since the previous purchase")
plt.legend()
plt.show()

In [ ]:
# first purchases (no previous purchase at all) vs the rest
first_purchase = df_eda["prior_purchase_count"] == 0
print(f"First purchases: {first_purchase.sum()} rows, repeat rate {df_eda.loc[first_purchase, target_col].mean():.3f}")
print(f"Other purchases: {(~first_purchase).sum()} rows, repeat rate {df_eda.loc[~first_purchase, target_col].mean():.3f}")

In [ ]:
# frequency: repeat rate by number of purchases in the previous 90 days
freq = df_eda["purchase_count_90d"].clip(upper=6).astype("Int64").astype(str).replace({"6": "6+"})
df_eda.groupby(freq)[target_col].agg(["mean", "size"]).rename(columns={"mean": "repeat rate", "size": "rows"}).round(3)

* There is a **clear break at 90 days**: clients whose previous purchase was less than 90 days ago come back in 66% to 75% of the cases, after 90 days the rate falls to around 40%, and to 29% after one year. The relation is not linear, so `log1p` or bins can help a linear model.
* **First purchases** (no previous purchase at all) come back in only **33%** of the cases, close to the clients whose previous purchase was more than one year ago. The "none / missing" bar of the plot (34%) is now almost only these first purchases, because the other missing values of `days_since_previous_purchase` were recovered in 3.6.7. This is why we want a flag `is_first_purchase` in Part 2 instead of just filling their missing values.
* **Frequency is even clearer**: with no purchase in the previous 90 days the repeat rate is 38%, with one purchase 57%, with three 83% and with six or more 99%.

#### **7.4. Rhythm of each Client**

The same number of days does not mean the same for every client: 30 days is a lot for a client who buys every week and very little for a client who buys once a year. So we test a ratio that compares the days since the previous purchase with the usual gap of the client: `days_since_previous_purchase / prior_mean_gap_days`. A value below 1 means this purchase came sooner than usual, and above 1 later than usual.

In [ ]:
rhythm = df_eda["days_since_previous_purchase"] / df_eda["prior_mean_gap_days"].replace(0, np.nan)
pair = pd.DataFrame({"rhythm": rhythm, target_col: df_eda[target_col]}).dropna()

rho, _ = stats.spearmanr(pair["rhythm"], pair[target_col])
print(f"Spearman correlation with the target: {rho:.3f}")
pair.groupby(target_col)["rhythm"].describe()[["count", "25%", "50%", "75%"]].round(2)

In [ ]:
# share of 0 and 1 in each group of the ratio
pair["rhythm group"] = pd.qcut(pair["rhythm"], 5, labels=["very early\n(< 0.26)", "early\n(0.26-0.62)",
                                                           "on time\n(0.62-1.04)", "late\n(1.04-1.75)",
                                                           "very late\n(> 1.75)"])
shares = pd.crosstab(pair["rhythm group"], pair[target_col], normalize="index") * 100
rows = pair["rhythm group"].value_counts().reindex(shares.index)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(shares.index[::-1], shares[0][::-1], color="tab:orange", label="0 · no repeat")
ax.barh(shares.index[::-1], shares[1][::-1], left=shares[0][::-1], color="tab:blue", label="1 · repeat")
for i, (zero, one, n) in enumerate(zip(shares[0][::-1], shares[1][::-1], rows[::-1])):
    ax.text(zero / 2, i, f"{zero:.0f}%", va="center", ha="center", color="white", fontsize=9)
    ax.text(zero + one / 2, i, f"{one:.0f}%", va="center", ha="center", color="white", fontsize=9)
    ax.text(101, i, f"n={n}", va="center", fontsize=8)
ax.axvline(100 - 100 * pair[target_col].mean(), color="black", linestyle="--", linewidth=1, label="average split")
ax.set(xlim=(0, 110), xlabel="% of purchases", title="Target split by rhythm ratio")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

The ratio is very similar in the two classes (median 0.92 for class 0 and 0.77 for class 1, Spearman −0.05). Only the "early" group (clients who came back at about half of their usual gap) stands out, with 68% of repeats against about 60% on average. The other groups are close to the average. So the idea makes sense, but alone the ratio has much less signal than the recency and the frequency. We keep it only as a candidate variable for Part 2, and the feature selection will decide.

#### **7.5. Categorical Variables**

For two categorical variables we use the **chi-square test** and the **Cramér's V** (from 0 = no association to 1 = perfect association). The chi-square is only reliable when the expected count of every cell is around 5 or more, so we also show the minimum expected count.

In [ ]:
cat_rows = []
for col in cat_cols:
    x = df_eda[col].fillna("<missing>") # the missing values are transformed in a string "<miaaing>" to not lose them
    table = pd.crosstab(x, df_eda[target_col])
    chi2, p_value, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    cat_rows.append({"feature": col, "levels": table.shape[0], "chi2": chi2, "p-value": p_value,
                     "Cramer's V": v, "min expected count": expected.min()})
pd.DataFrame(cat_rows).set_index("feature").sort_values("Cramer's V", ascending=False).round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, ["payment_type", "commercial_zone", "salesperson"]):
    by_level = df_eda.groupby(df_eda[col].fillna("<missing>"))[target_col].agg(["mean", "size"])
    by_level = by_level[by_level["size"] >= 30].sort_values("mean")
    ax.barh(by_level.index, by_level["mean"])
    ax.axvline(df_eda[target_col].mean(), color="orange", linestyle="--")
    ax.set(xlim=(0, 1), title=f"{col} (levels with >= 30 rows)", xlabel="repeat rate")
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

* `transport_method`, `payment_type` and `salesperson` have a moderate association with the target (Cramér's V between 0.20 and 0.29). Some payment types have repeat rates far from the average, which makes sense: the payment terms can show the type of relationship with the client.
* The V of `transport_method` is partly high **because** it has 80 levels: many tiny levels with extreme rates push the V up (their expected counts are far below 5). Grouping the rare levels into "Other" in Part 2 is what makes this variable usable.
* `commercial_zone` and `warehouse` have a weak association, and `document_series` has none (V ≈ 0, p = 0.97).

### **8. Redundancy between Variables**

Two variables can both be related to the target and still say the same thing. To check this we use the Spearman correlation between all the numeric variables and list the pairs above 0.90.

In [ ]:
corr = df_eda[numeric_cols].corr(method="spearman")

# only the lower triangle (the upper one is the same), without the empty first row and last column
corr_plot = corr.iloc[1:, :-1]
mask = np.triu(np.ones_like(corr_plot, dtype=bool), k=1)

# write the value only in the strong correlations
strong = 0.75
annot = corr_plot.round(2).astype(str).where(corr_plot.abs() >= strong, "")

fig, ax = plt.subplots(figsize=(14, 11), facecolor="white")
ax.set_facecolor("white")
sns.heatmap(corr_plot, mask=mask, annot=annot, fmt="", annot_kws={"size": 7}, cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, linewidths=0.5, linecolor="#EEEEEE",
            cbar_kws={"shrink": 0.6, "label": "Spearman rho"}, ax=ax)
ax.set_title(f"Spearman correlation between numeric variables (values shown when |rho| >= {strong})")
plt.tight_layout()
plt.show()

In [ ]:
pairs = corr.where(np.tril(np.ones_like(corr, dtype=bool), k=-1)).stack().reset_index()
pairs.columns = ["variable A", "variable B", "Spearman rho"]
pairs = pairs.reindex(pairs["Spearman rho"].abs().sort_values(ascending=False).index)
pairs[pairs["Spearman rho"].abs() >= 0.90].round(3).reset_index(drop=True)

In [ ]:
# some links come from how the columns are built
no_recent = df_eda["purchase_count_90d"] == 0
print(f"Rows with no purchase in the last 90 days and days_since_previous_purchase > 90: "
      f"{(df_eda.loc[no_recent, 'days_since_previous_purchase'].dropna() > 90).mean():.1%}")

Eight pairs are above 0.90:
* `Total a Pagar`, `Total Bruto` and `Total Liquido` (0.99 to 1.00): they are linked by the accounting rules (net = gross − discounts), so one of them is enough.
* The count and the spend of the short windows: `purchase_count_30d` with `spend_30d` (0.97) and `purchase_count_90d` with `spend_90d` (0.91).
* `prior_purchase_count` with `prior_total_spend` (0.90), `prior_max_purchase_value` with `prior_std_purchase_value` (0.94) and `spend_365d` with `spend_180d` (0.90).

Redundancy is a problem mainly for the linear model (unstable coefficients), so in Part 3 we will use a redundancy filter and then the other feature selection methods.

### **9. The Clients Table (clientes.csv)**

The guidelines ask us to decide if and how to use `clientes.csv` through `customer_id`. First we check the quality of the table, then if it adds something new.

In [ ]:
print(f"Clients in clientes.csv: {len(df_clientes):,} | customer_id unique: {df_clientes['customer_id'].is_unique}")
print(f"Train clients found in clientes.csv: {df['customer_id'].isin(df_clientes['customer_id']).mean():.0%}")

# we only look at our clients, the other rows of the table never reach the model
clientes_train = df_clientes[df_clientes["customer_id"].isin(df["customer_id"])]
print(f"Train clients described: {len(clientes_train)}")

pd.DataFrame({
    "% missing": (100 * clientes_train.isna().mean()).round(1),
    "distinct values": clientes_train.nunique(),
    "distinct / clients": (clientes_train.nunique() / len(clientes_train)).round(3),
})

* `customer_id` is a good key: it is unique in the table and every client of the train is there. The table has 1,143 clients and the train only 607, but this is not a problem if we do a **left join** from the purchases (each purchase gets its client and the number of rows does not change) and we do not compute any statistic on the whole table.
* `customer_type` is **100% empty**.
* `company_name`, `address`, `street`, `postal_code` and `city` are almost unique for each client, so they identify the client instead of describing him (the same problem as using `customer_id`). Some of them are also half empty.
* The possible variables are the geography (`country_or_region` 7 levels, `commercial_region` 32, `island` 10, `municipality` 150, `postal_area` 188) and `customer_since_year`.

In [ ]:
# join the candidate columns to each purchase and measure the association with the target
candidates = ["country_or_region", "commercial_region", "island", "municipality", "customer_since_year"]
joined = df_eda.join(df_clientes.set_index("customer_id")[candidates], on="customer_id")

rows = []
for col in candidates:
    x = joined[col].astype("string").fillna("<missing>")
    table = pd.crosstab(x, joined[target_col])
    chi2, p_value, _, expected = chi2_contingency(table, correction=False)
    rows.append({"feature": col, "levels": table.shape[0],
                 "Cramer's V": np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1))),
                 "min expected count": expected.min()})
pd.DataFrame(rows).set_index("feature").round(3)

In [ ]:
# is customer_since_year new information, or the same as customer_tenure_days?
purchase_year = pd.to_datetime(joined["purchase_date"]).dt.year
years_as_client = purchase_year - joined["customer_since_year"]
rho, _ = stats.spearmanr(years_as_client, joined["customer_tenure_days"], nan_policy="omit")
print(f"Spearman(purchase year - customer_since_year, customer_tenure_days) = {rho:.3f}")
print(f"Purchases made before the customer_since_year: {(years_as_client < 0).mean():.1%}")

When we look at `customer_since_year`, 2015 has much more clients than the other years. Since the data only starts in July 2015, lets check if these are really new clients or old clients that were loaded into the system when it started.

In [ ]:
# how many clients have 2015 as start year, inside and outside the train?
in_train = df_clientes["customer_id"].isin(df["customer_id"])
since_2015 = df_clientes["customer_since_year"] == 2015
print(f"Clients with customer_since_year = 2015: {since_2015.sum()} of {len(df_clientes)}")
print(f"  clients in train: {since_2015[in_train].mean():.0%} ({since_2015[in_train].sum()} of {in_train.sum()})")
print(f"  clients not in train: {since_2015[~in_train].mean():.0%} ({since_2015[~in_train].sum()} of {(~in_train).sum()})")

# first purchases in 2015 vs first purchases in the other years
first_purchase = df_eda["prior_purchase_count"] == 0
year = pd.to_datetime(df_eda["purchase_date"]).dt.year
print(f"\nRepeat rate of first purchases in 2015: {df_eda.loc[first_purchase & (year == 2015), target_col].mean():.2f}")
print(f"Repeat rate of first purchases after 2015: {df_eda.loc[first_purchase & (year > 2015), target_col].mean():.2f}")

More than half of all the clients (664 of 1,143) have 2015 as start year, while in the other years only 13 to 82 clients join per year. Most of these 2015 clients never buy anything in our data (79% of the clients that are not in the train are from 2015). Also, the "first purchases" of 2015 come back much more than the first purchases of the other years (45% vs 29%), so they look like old clients and not new ones.

So it looks like the company started this system in 2015 and loaded all the clients it already had, but not their old purchases. This means that in 2015 the history variables (`prior_purchase_count`, `customer_tenure_days`...) are lower than the real history of these clients, and some "first purchases" are not really first purchases. So the purchases of 2015 can teach the model something wrong: clients that look new but behave like loyal clients. In Part 2 we will test if the model gets better when we remove the 2015 purchases from the training data, or when we add a flag for this period.

In [ ]:
# is a missing island related to the target?
island_missing = joined["island"].isna()
print(f"Island missing: {island_missing.sum()} purchases ({joined.loc[island_missing, 'customer_id'].nunique()} clients), repeat rate {joined.loc[island_missing, target_col].mean():.2f}")
print(f"Island known: {(~island_missing).sum()} purchases ({joined.loc[~island_missing, 'customer_id'].nunique()} clients), repeat rate {joined.loc[~island_missing, target_col].mean():.2f}")

`island` is missing for almost half of the clients, and these clients come back more (64% vs 52%). Lets check where these clients are, to understand what a missing island means.

In [ ]:
# where are the clients with and without island?
island_known = clientes_train["island"].notna()
country = clientes_train["country_or_region"].str.split(":").str[0]
region = clientes_train["commercial_region"].str.split(":").str[0]

print("Regions of the clients with island:", region[island_known].value_counts().to_dict())
island_regions = region.isin(["Region RG17", "Region RG29"])
print("Clients in RG17 or RG29 without island:", (island_regions & ~island_known).sum())

pd.crosstab(country, island_known)

All the clients with an island are from `Country CT10` and from only two regions (RG17 and RG29), and every client of these two regions has the island filled. The clients without island are the other regions of CT10 (the mainland) and the other countries. So a missing island is not a data error: it means the client is not on an island. In Part 2 we keep it as its own category ("Missing" = not on an island) and do not fill it with the most frequent island, which would put mainland clients on an island.

This also shows that `island` is a detail inside `commercial_region`: the island clients are exactly the clients of RG17 and RG29. So the two variables are partly redundant, and the feature selection will decide if both are needed.

In [ ]:
# repeat rate when each column of clientes.csv is missing or not
all_cols = df_clientes.columns.drop(["customer_id", "customer_type"])
joined_all = df_eda.join(df_clientes.set_index("customer_id")[all_cols], on="customer_id")

rows = []
for col in all_cols:
    missing = joined_all[col].isna()
    if missing.sum() == 0:
        continue
    rows.append({"column": col,
                 "clients missing": joined_all.loc[missing, "customer_id"].nunique(),
                 "repeat rate (missing)": joined_all.loc[missing, target_col].mean(),
                 "repeat rate (known)": joined_all.loc[~missing, target_col].mean()})

both = clientes_train["municipality"].isna() & clientes_train["postal_area"].isna()
print(f"Clients with municipality and postal_area both missing: {both.sum()}")
pd.DataFrame(rows).set_index("column").sort_values("clients missing", ascending=False).round(2)

Besides `island`, the clients without `municipality` or `postal_area` come back much less (38-41% vs 57%). These are almost the same clients in both columns (38 clients with both missing), so it looks like a group of clients with an incomplete address. We do not use these two columns (too detailed), but in Part 2 we can test a simple flag "incomplete address". It is only about 40 clients, so this signal is weak and the feature selection will decide. `commercial_region` is missing for only 12 clients, which is too few to say anything.

**Decision on `clientes.csv` (to do in Part 2)**
* **We use only the geography that is not too detailed**: `country_or_region`, `commercial_region` and `island` (missing → "Missing", rare levels → "Other"). They describe where the client is, which can matter for logistics and sales. Their association with the target goes from weak (`country_or_region`, V = 0.07) to moderate (`commercial_region`, V = 0.26), but part of this last one comes from its 33 levels with few clients, like we saw with `transport_method`. The feature selection will decide if they stay.
* **We do not use** `customer_type` (because it is empty), `company_name`, `address`, `street`, `postal_code` and `city` (they work like identifiers), and `municipality` and `postal_area`: 151 and 189 levels for 607 clients is too detailed, and the high V of `municipality` (0.37) comes from the few clients in each municipality, not from a real pattern.  Their missing values, however, can still be useful as a flag.
* **We do not use** `customer_since_year`: it has a correlation of 0.92 with `customer_tenure_days`, which is already in the train and is measured in days. The two tables are consistent (no purchase before the year the client joined), so the year does not add anything new.  Its biggest pattern (the 2015 peak) comes from the clients loaded into the system, not from real new clients.
* **One thing we can not check**: we do not know if `clientes.csv` shows the clients as they are today or as they were at the time of each purchase. If a client changed region, for example, a purchase from 2016 would get the region of today, which is information the company did not have in 2016. Since the geography of a client rarely changes we accept this risk, but it is one more reason to use only the less detailed geography.

A geography that is too detailed is almost an identifier: with only 607 clients, a municipality with two clients is almost the same as the client ID.

### **10. Identifiers and Random Noise**

The guidelines say that variables that look like identifiers should be treated critically. We check `ID`, `customer_id` and `random_noise` against the target and against the date.

In [ ]:
id_frame = df.reset_index()
id_dates = pd.to_datetime(id_frame["purchase_date"]).astype("int64")

pd.DataFrame({
    "Spearman with target": [stats.spearmanr(id_frame[c], id_frame[target_col])[0] for c in ["ID", "customer_id", "random_noise"]],
    "Spearman with date": [stats.spearmanr(id_frame[c], id_dates)[0] for c in ["ID", "customer_id", "random_noise"]],
}, index=["ID", "customer_id", "random_noise"]).round(3)

In [ ]:
# why would customer_id be related to the target? Lets look at the clients with more purchases
top_clients = df["customer_id"].value_counts().head(5).index
(df[df["customer_id"].isin(top_clients)]
 .groupby("customer_id")[target_col].agg(["size", "mean"])
 .rename(columns={"size": "purchases", "mean": "repeat rate"})
 .sort_values("purchases", ascending=False).round(3))

* **`ID`** is just the number of the row: it has no relation with the target or with the date. It stays as the index and is never used as a variable.
* **`customer_id`** has a small correlation with the target (−0.12), but this is a **spurious relation**: the number is just a code, and the correlation only appears because a few very active clients (hundreds of purchases and repeat rates close to 1) have specific ID values. A model could learn these ID values and memorise these clients, which does not work for new purchases. So it is not used as a variable, only as the key to join `clientes.csv`.
* **`random_noise`** has no relation with anything (correlation ≈ 0 with the target and with the date). We do **not** delete it by hand: we keep it as a control in the feature selection (Part 3). A good feature selection method should remove it by itself.

### **11. Main Findings**

What we've discovered in this first part, and what we will do with it in Part 2:

1. **Target and floor**: 56% of the purchases are followed by another purchase in the next 90 days. Predicting always 1 already gives **F1 = 0.72**, so this is the floor every model has to beat, and the decision threshold will have to be tuned and not left at 0.5.
2. **Data quality**: 65 empty rows (drop) and impossible values in five columns. `prior_mean_purchase_value` can be recomputed as total / count for every row, while the wrong values of `spend_365d` (69), the gaps of -4 days (32) and the impossible maximums (46) become missing. Many missing values can be computed exactly from other columns (516 cells, 3.6.7), so they are recovered before any imputation. The invoice columns are consistent, and a cost of 0 (486 purchases) is possible, so it is kept.
3. **Missing values are mostly structural**: first purchases have no history. We will add a flag `is_first_purchase` and missing indicators, and impute with the train median (never with 0). The context columns (about 3% missing each) get a "Missing" category.
4. **Outliers are our best clients**: we keep them and use `log1p` for the models that need it.
5. **Recency and frequency are the strongest signals** (Spearman up to 0.44: `purchase_count_365d`, `prior_mean_gap_days`, `purchase_count_180d`, `purchase_count_90d`). The relations are monotonic but not linear (for example the break at 90 days), which is why we use Spearman. The values of the current invoice matter very little, and first purchases come back much less (33% vs 58%).
6. **Categorical variables**: `transport_method`, `payment_type` and `salesperson` have a moderate association (Cramér's V 0.20 to 0.29), but their rare levels have to be grouped into "Other". `document_series` has no relation with the target.
7. **Redundancy**: 8 pairs of variables have a correlation above 0.90 (for example `Total a Pagar`, `Total Bruto` and `Total Liquido`), so in Part 3 we start with a redundancy filter. `random_noise` stays as a control for the feature selection, and `ID` and `customer_id` are never used as variables.
8. **clientes.csv**: left join by `customer_id`, using only `country_or_region`, `commercial_region` and `island`. A missing `island` means the client is not on an island (it is only filled in regions RG17 and RG29), so it becomes its own category. Clients with an incomplete address (`municipality` and `postal_area` missing) come back less, which we can test as a flag.
9. **Time**: the repeat rate changes from year to year (0.48 to 0.63) and the cumulative variables grow with time, and the purchases of the same client are not independent. So the validation will be **temporal** (train on the past, validate on the most recent purchases) and the year is not used as a variable. The clients registered in 2015 look like old clients loaded into the system without their past purchases, so in Part 2 we will test the model with and without the 2015 purchases.